In [1]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

from pathlib import Path
from copy import deepcopy
import multiprocessing as mp
import logging
import random
from itertools import cycle
import os
from functools import partial

import mlflow
import matplotlib.pyplot as plt
from networkx import number_weakly_connected_components
import torch
import numpy as np
from dotenv import load_dotenv

from halcyon.ModelRunner import run_training_job, ModelRunner
from halcyon.Simulators import NoiseLevel, SimulationResult

from halcyon.Models import PhoenixAddAndProdLayers, MLP, CellBox
from halcyon.Models.Losses import MSELossPlusL1


seed = 174
# for CPU
torch.manual_seed(seed)              
# for all GPUs
torch.cuda.manual_seed_all(seed)     
np.random.seed(seed) 
random.seed(seed) 

load_dotenv() 
logging.basicConfig(
    level=logging.INFO, 
    format='%(asctime)s - %(levelname)s - %(message)s',
    force=True  # Important in Jupyter to reset config
)


Check if MLFlow connects correctly

In [2]:
print(os.getenv("MLFLOW_TRACKING_URI"))

https://mlflow.cropresilience.org


In [3]:
[f"{experiment.name=}" for experiment in mlflow.search_experiments()]
mlflow.set_experiment("Halcyon comparing models and datasets")
mlflow.set_experiment_tag("Tool name", "HALCYON") 

# Input data

## Downloading data from MLFlow

In [4]:
sim_run_id_to_use = "912f193730964b508691ee4ec81d66be"  
run = mlflow.get_run(sim_run_id_to_use)

params = run.data.params
print(params)

{'nr_genes': '20', 'sparsity': '0.1', 'num_modules': '2', 'noise_level': 'LOW', 'seed': '174'}


In [5]:
ml_flow_download_path = mlflow.artifacts.download_artifacts(run_id=sim_run_id_to_use, artifact_path="dataset")
sim_output = SimulationResult.load_from_pkl(Path(ml_flow_download_path) / "simulation_result.pkl")

2026-03-23 15:01:32,615 - WARNING - Connection pool is full, discarding connection: mlflow.cropresilience.org. Connection pool size: 10
2026-03-23 15:01:32,628 - WARNING - Connection pool is full, discarding connection: mlflow.cropresilience.org. Connection pool size: 10
2026-03-23 15:01:32,642 - WARNING - Connection pool is full, discarding connection: mlflow.cropresilience.org. Connection pool size: 10
2026-03-23 15:01:32,655 - WARNING - Connection pool is full, discarding connection: mlflow.cropresilience.org. Connection pool size: 10
2026-03-23 15:01:32,668 - WARNING - Connection pool is full, discarding connection: mlflow.cropresilience.org. Connection pool size: 10
2026-03-23 15:01:33,766 - WARNING - Connection pool is full, discarding connection: mlflow.cropresilience.org. Connection pool size: 10
2026-03-23 15:01:33,772 - WARNING - Connection pool is full, discarding connection: mlflow.cropresilience.org. Connection pool size: 10
2026-03-23 15:01:33,785 - WARNING - Connection p

## Convert sim output into datasets for use with the trainer object

In [6]:
# From sim output to datasets
single_ko_dataset, double_ko_dataset, time_scaling_factor = sim_output.create_datasets()
train_ko_dataset, validation_ko_dataset = single_ko_dataset.split_train_test(test_fraction=0.25)

train_dataset_name = "train_single_ko"
val_dataset_name ="val_single_ko"
test_dataset_name = "double_ko"
dataset_dict = {
    train_dataset_name: train_ko_dataset,
    val_dataset_name: validation_ko_dataset,
    test_dataset_name: double_ko_dataset,}

for name, dataset in dataset_dict.items():
    print(f"Length {name}: {len(dataset)}")

Length train_single_ko: 15
Length val_single_ko: 5
Length double_ko: 20


# Trainers (with different models) creation

In [7]:
train_jobs : list[ModelRunner] = []
device = cycle(['cuda:0', 'cuda:1'])
batch_size = 32
nr_epochs = 100

## Phoenix

In [8]:
for hidden_size in [20]:
    for do_only_pos in [True]:        
        for lambda_value in [0., 0.1, 1]:
            loss = MSELossPlusL1(lambda_param=lambda_value)
            ml_model = PhoenixAddAndProdLayers(
                        modelled_gene_labels=single_ko_dataset.genes,
                        hidden_size=hidden_size,
                        condition_names=single_ko_dataset.all_environments,
                        synthesis_only_positive=do_only_pos,
                        device=next(device)
                )

            # my_loss = MSELossPlusMachineLearningL2(lambda_param=0.5)
            model_runner = ModelRunner(
                model = ml_model,
                batch_size=batch_size,
                optimizer_kwargs = {'lr': 0.01},
                loss=loss
            )
            train_jobs.append(model_runner)

## CellBox

In [ ]:
cellbox = CellBox(
            modelled_gene_labels=single_ko_dataset.genes,
            condition_names=single_ko_dataset.all_environments,
            device=next(device)
    )

model_runner = ModelRunner(
    model = cellbox,
    batch_size=batch_size,
    optimizer_kwargs = {'lr': 0.01}
)
train_jobs.append(model_runner)

## MLP

In [ ]:
mlp = MLP(
        modelled_gene_labels=single_ko_dataset.genes,
        condition_names=single_ko_dataset.all_environments,
        hidden_size=5,
        nr_layers=2,
        device=next(device)
)

model_runner = ModelRunner(
    model = mlp,
    batch_size=batch_size,
    optimizer_kwargs = {'lr': 0.01}
)
train_jobs.append(model_runner)

In [9]:
for job in train_jobs:
    print(job)
    print()

Modelrunner (Model=PhoenixAddAndProdLayers with hyperparams: ['device=cuda:0', 'hidden_size=20', 'synthesis_only_positive=True', 'disable_per_gene_multipliers=False'],
 optimizer=Adam,
 optimizer_kwargs={'lr': 0.01},
 batch_size=32,
 loss_fn=MSELossPlusL1 with lambda=0.0
)

Modelrunner (Model=PhoenixAddAndProdLayers with hyperparams: ['device=cuda:1', 'hidden_size=20', 'synthesis_only_positive=True', 'disable_per_gene_multipliers=False'],
 optimizer=Adam,
 optimizer_kwargs={'lr': 0.01},
 batch_size=32,
 loss_fn=MSELossPlusL1 with lambda=0.1
)

Modelrunner (Model=PhoenixAddAndProdLayers with hyperparams: ['device=cuda:0', 'hidden_size=20', 'synthesis_only_positive=True', 'disable_per_gene_multipliers=False'],
 optimizer=Adam,
 optimizer_kwargs={'lr': 0.01},
 batch_size=32,
 loss_fn=MSELossPlusL1 with lambda=1
)



In [12]:
train_jobs[2]

Modelrunner (Model=PhoenixAddAndProdLayers with hyperparams: ['device=cuda:0', 'hidden_size=20', 'synthesis_only_positive=True', 'disable_per_gene_multipliers=False'],
 optimizer=Adam,
 optimizer_kwargs={'lr': 0.01},
 batch_size=32,
 loss_fn=MSELossPlusL1 with lambda=1
)

In [13]:
train_jobs[2].train_and_predict_for_mlflow(dataset_dict=dataset_dict, 
                                           train_data_key_in_dict=train_dataset_name,
                                           val_data_key_in_dict=val_dataset_name,
                                           test_data_key_in_dict=test_dataset_name,
                                           nr_epochs=nr_epochs,
                                           simulation_parent_id=sim_run_id_to_use,
                                           save_every_n_epochs=10)

2026-03-23 15:06:58,370 [MainProcess] Epoch 0, train_loss	0.1299314945936203
2026-03-23 15:06:58,865 [MainProcess] Epoch 0, val_loss	0.1053723618388176
2026-03-23 15:06:59,208 [MainProcess] Epoch 0, val_mse	0.08289209753274918
2026-03-23 15:06:59,910 [MainProcess] Epoch 0, test_loss	0.08623120933771133
2026-03-23 15:07:00,748 [MainProcess] Epoch 0, test_mse	0.06375093758106232
2026-03-23 15:07:01,453 [MainProcess] Epoch 1, train_loss	0.11429790407419205
2026-03-23 15:07:01,944 [MainProcess] Epoch 1, val_loss	0.08551663160324097
2026-03-23 15:07:02,299 [MainProcess] Epoch 1, val_mse	0.06445857882499695
2026-03-23 15:07:03,007 [MainProcess] Epoch 1, test_loss	0.08325603604316711
2026-03-23 15:07:03,718 [MainProcess] Epoch 1, test_mse	0.06219799816608429
2026-03-23 15:07:04,421 [MainProcess] Epoch 2, train_loss	0.09501489251852036
2026-03-23 15:07:04,948 [MainProcess] Epoch 2, val_loss	0.07453063130378723
2026-03-23 15:07:05,318 [MainProcess] Epoch 2, val_mse	0.05127084255218506
2026-03-2

🏃 View run PhoenixAddAndProdLayers at: https://mlflow.cropresilience.org/#/experiments/4/runs/fe82b22699b54151ac8a04b904585e81
🧪 View experiment at: https://mlflow.cropresilience.org/#/experiments/4


KeyboardInterrupt: 

# Do training

In [ ]:
# mp.set_start_method("spawn", force=True)
# with mp.Pool(processes=4) as pool:
#     pool.map(
#         partial(
#             run_training_job, 
#             dataset_dict=dataset_dict, 
#             train_data_key_in_dict=train_dataset_name, 
#             simulation_parent_id=sim_run_id_to_use, 
#             do_mlflow_model_logging=False,
#         ),
#         train_jobs
#     )

# All files have now been logged to MLFlow

To see how to process this data further look into the MLFlow loading and plotting notebook(s)